In [2]:
import requests
import pandas as pd
from IPython.display import HTML

In [10]:
import requests
import pandas as pd
from IPython.display import HTML

BASE_URL = "https://pokeapi.co/api/v2"

def pegar_url_tipo(tipo):
    """Retorna uma URL estável SVG/PNG para os badges dos tipos."""
    return f"https://raw.githubusercontent.com/duiker101/pokemon-type-svg-icons/master/icons/{tipo}.svg"

def pegar_icone_fraquezas(types):
    todos_tipos = [
        'normal', 'fire', 'water', 'grass', 'electric', 'ice', 
        'fighting', 'poison', 'ground', 'flying', 'psychic', 'bug', 
        'rock', 'ghost', 'dragon', 'steel', 'dark', 'fairy'
    ]
    multiplicador = {t: 1.0 for t in todos_tipos}
    
    for type_name in types:
        type_res = requests.get(f"{BASE_URL}/type/{type_name}")
        if type_res.status_code == 200:
            rel = type_res.json()['damage_relations']
            for t in rel['double_damage_from']:
                multiplicador[t['name']] *= 2.0
            for t in rel['half_damage_from']:
                multiplicador[t['name']] *= 0.5
            for t in rel['no_damage_from']:
                multiplicador[t['name']] *= 0.0
                
    badges = []
    sorted_weaknesses = sorted(
        [(t, mult) for t, mult in multiplicador.items() if mult > 1.0], 
        key=lambda x: x[1], 
        reverse=True
    )
    
    for w_name, mult in sorted_weaknesses:
        badge_url = pegar_url_tipo(w_name)
        
        # String em uma única linha (sem \n)
        if mult == 4.0:
            badge_html = f'<span style="display: inline-block; border: 2px solid #ff0000; border-radius: 6px; padding: 2px 4px; margin: 2px; background-color: #ffe6e6;"><img src="{badge_url}" height="18" width="18" style="vertical-align: middle;" title="Fraqueza 4x: {w_name}"/> <b style="color: #cc0000; font-size: 11px; vertical-align: middle;">4x</b></span>'
        else:
            badge_html = f'<span style="display: inline-block; border: 1px solid #ccc; border-radius: 6px; padding: 2px; margin: 2px; background-color: #f9f9f9;"><img src="{badge_url}" height="18" width="18" style="vertical-align: middle;" title="Fraqueza 2x: {w_name}"/></span>'
        
        badges.append(badge_html)
        
    return "".join(badges) if badges else "Nenhuma"


def get_pokemon_card_mini_html(pokemon_name):
    """Busca a sprite e cria um mini card HTML sem quebras de linha."""
    res = requests.get(f"{BASE_URL}/pokemon/{pokemon_name.lower()}")
    if res.status_code != 200:
        return f"<span>{pokemon_name.capitalize()}</span>"
    
    data = res.json()
    sprite_url = data['sprites']['front_default']
    
    # String contínua sem \n
    return f'<div style="display: inline-block; text-align: center; margin: 0 4px;"><img src="{sprite_url}" width="35" height="35" style="display: block; margin: 0 auto;"/><span style="font-size: 10px;">{pokemon_name.capitalize()}</span></div>'


def get_evolution_info_html(pokemon_name):
    species_res = requests.get(f"{BASE_URL}/pokemon-species/{pokemon_name}")
    if species_res.status_code != 200:
        return "-", "-"
    
    species_data = species_res.json()
    
    # 1. Evolução Anterior
    prev_evt = species_data.get('evolves_from_species')
    prev_html = get_pokemon_card_mini_html(prev_evt['name']) if prev_evt else "-"
    
    # 2. Evoluções Posteriores
    chain_url = species_data['evolution_chain']['url']
    chain_res = requests.get(chain_url)
    if chain_res.status_code != 200:
        return prev_html, "-"
    
    chain_data = chain_res.json()['chain']
    next_names = []
    
    def find_next_evolutions(node):
        if node['species']['name'] == pokemon_name:
            for evo in node['evolves_to']:
                next_names.append(evo['species']['name'])
        else:
            for evo in node['evolves_to']:
                find_next_evolutions(evo)
                
    find_next_evolutions(chain_data)
    
    if next_names:
        next_html = "".join([get_pokemon_card_mini_html(name) for name in next_names])
    else:
        next_html = "-"
    
    return prev_html, next_html


def get_pokemon_advanced_row(name_or_id):
    res = requests.get(f"{BASE_URL}/pokemon/{str(name_or_id).lower()}")
    if res.status_code != 200:
        return None
    
    data = res.json()
    pokemon_name = data['name']
    
    # Sprite principal
    sprite_url = data['sprites']['front_default']
    img_tag = f'<img src="{sprite_url}" width="60" />'
    
    # Badges dos tipos (SVG limpos)
    types = [t['type']['name'] for t in data['types']]
    types_badges = []
    for t in types:
        badge_url = pegar_url_tipo(t)
        types_badges.append(f'<span style="display: inline-block; border: 1px solid #ddd; border-radius: 4px; padding: 2px; margin-right: 2px;"><img src="{badge_url}" height="20" width="20" title="{t}"/></span>')
    badges_tag = "".join(types_badges)
    
    # Badges de fraquezas
    weaknesses_tag = pegar_icone_fraquezas(types)
    
    # Evoluções
    prev_evo_html, next_evo_html = get_evolution_info_html(pokemon_name)
    
    return {
        "ID": f"#{data['id']:03d}",
        "Imagem": img_tag,
        "Nome": pokemon_name.capitalize(),
        "Tipos": badges_tag,
        "Fraquezas": weaknesses_tag,
        "Evolução Anterior": prev_evo_html,
        "Evolução Posterior": next_evo_html
    }

In [12]:
# Pokémons para teste (incluindo Eevee que possui múltiplas evoluções e Scizor/Charizard com fraqueza 4x)
lista_pokemons = ["charizard", "scizor", "eevee", "pupitar","garchomp","yveltal", "gyarados"]

dados = [get_pokemon_advanced_row(p) for p in lista_pokemons if get_pokemon_advanced_row(p) is not None]

df = pd.DataFrame(dados)

# Renderização no Jupyter Notebook com alinhamento centralizado
HTML(df.to_html(escape=False, index=False))

ID,Imagem,Nome,Tipos,Fraquezas,Evolução Anterior,Evolução Posterior
#006,,Charizard,,4x,Charmeleon,-
#212,,Scizor,,4x,Scyther,-
#133,,Eevee,,,-,VaporeonJolteonFlareonEspeonUmbreonLeafeonGlaceonSylveon
#247,,Pupitar,,4x 4x,Larvitar,Tyranitar
#445,,Garchomp,,4x,Gabite,-
#717,,Yveltal,,,-,-
#130,,Gyarados,,4x,Magikarp,-
